<!-- field-paper-role-banner -->
> 🟦 **실습본** — TODO를 직접 구현하고 assertion을 통과시키세요. 정답본은 같은 위치의 필요한 부분만 확인합니다.

# 자기지도 학습 · 멀티모달 · 08. BLIP: 이해·생성과 CapFilt

## BLIP: Bootstrapping Language-Image Pre-training for Unified Vision-Language Understanding and Generation

- **원 논문:** [BLIP: Bootstrapping Language-Image Pre-training for Unified Vision-Language Understanding and Generation](https://arxiv.org/abs/2201.12086)
- **저자 / 발표:** Junnan Li, Dongxu Li, Caiming Xiong, Steven Hoi · ICML (2022)
- **난이도 / 예상 시간:** 고급 · 약 85분
- **선수 지식:** CLIP, image-text matching, language modeling
- **로컬 학습 데이터:** `data/field_curriculum/multimodal_pairs.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** MED의 ITC·ITM·LM 세 목적과 CapFilt의 생성/필터 단계를 작은 paired corpus에서 재현합니다.

**축소하거나 생략한 부분:** 1.29억 web pairs·ViT/BERT 대신 240개 pair와 작은 shared text embedding을 사용합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1, Figure 2 | ITC·ITM·LM 세 동작 모드 | 세 loss 유한값 |
| §3.1 Image-Text Contrastive | unimodal alignment | 대칭 CE |
| §3.1 Image-Text Matching/LM | negative match·causal token prediction | binary/next-token loss |
| §3.2, Figure 3 | captioner 생성+filter 제거 CapFilt | 정제 전후 noise |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(17)

DATA_PATH = Path("data/field_curriculum/multimodal_pairs.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
images = torch.tensor(raw["images"], dtype=torch.float32)
tokens = torch.tensor(raw["tokens"], dtype=torch.long)
class_ids = torch.tensor(raw["class_ids"], dtype=torch.long)
sequences = torch.tensor(raw["sequences"], dtype=torch.float32)
train_idx = torch.tensor(raw["train_idx"], dtype=torch.long)
test_idx = torch.tensor(raw["test_idx"], dtype=torch.long)
assert len(images) == len(tokens) == len(class_ids) == len(sequences)
assert images.shape[1:] == (1, 8, 8) and tokens.ndim == 2 and sequences.ndim == 3
print(
    f"pairs={len(images)}, classes={class_ids.unique().numel()}, "
    f"image={tuple(images.shape[1:])}, tokens={tokens.shape[1]}"
)
print(ACCELERATOR.summary())
print("dataset/index preprocessing stays on CPU, while neural training uses DEVICE")

## 포트폴리오 해설: 표현, 목적함수, 업데이트

$$\mathcal L_{BLIP}=\mathcal L_{ITC}+\mathcal L_{ITM}+\mathcal L_{LM}$$

- **기호 정의:** ITC는 정렬, ITM은 pair 일치, LM은 다음 token 생성 목적입니다.
- **수식의 역할:** 하나의 모델에 이해와 생성을 결합하고 생성·필터 과정으로 caption을 정제합니다.
- **구현 이유:** representation 붕괴·modality 누설·잘못된 positive 연결을
  코드에서 확인할 수 있도록 핵심 수식을 view 준비와 loss 경계로 분리했습니다.
- **Task/코드 대응:** Task P의 `prepare_views`, 논문 전용 `nn.Module`의
  `forward`, `compute_loss`, `training_step`이 §3.1, Figure 2을 구현합니다.
- **입출력 shape:** image [B, 1, 8, 8], token [B, L] → contrastive [B, B], LM [B, L-1, V]
- **평가:** 세 loss, retrieval, clean/noisy pair filtering precision
- **원문 대비 한계:** 1.29억 web pairs·ViT/BERT 대신 240개 pair와 작은 shared text embedding을 사용합니다.

구현은 `로컬 데이터 통계 → view/modality 준비 → model → objective/update → 평가`
순서입니다. 이 순서는 데이터 누설과 stop-gradient 경계를 코드에서 찾기 쉽게 합니다.

### 구현 단계 2 · 합성 입력·데이터 계약 (data-contract 예외)

- **원문 위치:** 해당 없음 — 원 논문의 학습 연산이 아니라 외부 다운로드를 없애기 위한 축소 입력 생성 단계입니다.
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 논문 메커니즘과 데이터 규모 효과를 분리하고 Windows CPU에서도 같은 입력을 재현합니다.
- **완료 증거:** pair 정렬, batch 크기, dtype과 입력 rank assertion이 통과해야 합니다.

In [ ]:
image_features = torch.tensor(raw["image_features"], dtype=torch.float32)
class_image_features = torch.tensor(raw["class_image_features"], dtype=torch.float32)
class_tokens = torch.stack(
    [tokens[torch.where(class_ids == c)[0][0]] for c in class_ids.unique(sorted=True)]
)

### 구현 단계 3 · 합성 입력·데이터 계약 (data-contract 예외)

- **원문 위치:** 해당 없음 — 원 논문의 학습 연산이 아니라 외부 다운로드를 없애기 위한 축소 입력 생성 단계입니다.
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 논문 메커니즘과 데이터 규모 효과를 분리하고 Windows CPU에서도 같은 입력을 재현합니다.
- **완료 증거:** pair 정렬, batch 크기, dtype과 입력 rank assertion이 통과해야 합니다.

In [ ]:
# TODO 08-1: 35% noisy web caption과 nearest-prototype captioner를 만드세요.
def make_web_and_synthetic(text, image_feat, rate=0.35):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 4 · 핵심 연산·모델

- **원문 위치:** §3.2, Figure 3
- **이 셀의 책임:** captioner 생성+filter 제거 CapFilt
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 정제 전후 noise. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 08-2: captioner가 생성한 caption과 일치하는 web caption만 남기는 CapFilt를 구현하세요.
def capfilt(web: Tensor, synthetic: Tensor):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §3.1, Figure 2 / §3.1 Image-Text Contrastive / §3.1 Image-Text Matching/LM
- **이 셀의 책임:** ITC·ITM·LM 세 동작 모드 / unimodal alignment / negative match·causal token prediction
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 세 loss 유한값 / 대칭 CE / binary/next-token loss. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 08-3: shared text embedding을 쓰는 ITC·ITM·LM 세 loss를 구현하세요.
class TinyBLIP(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, vocab, dim=16):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def losses(self, x, t):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 6 · 평가·완료 증거

- **원문 위치:** §3.1, Figure 2 / §3.1 Image-Text Contrastive / §3.1 Image-Text Matching/LM / §3.2, Figure 3
- **이 셀의 책임:** ITC·ITM·LM 세 동작 모드 / unimodal alignment / negative match·causal token prediction / captioner 생성+filter 제거 CapFilt
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 세 loss 유한값 / 대칭 CE / binary/next-token loss / 정제 전후 noise. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 08-4: 세 목적을 공동 학습하고 정제 전후와 retrieval을 보고하세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

### Task P1 · modality/view 준비와 핵심 모델

image·text·sequence의 shape를 먼저 고정하고 augmentation 또는 modality
경계를 명시한 뒤 논문 전용 `nn.Module`을 구성합니다.

### 구현 단계 7 · 핵심 연산·모델

- **원문 위치:** §3.1, Figure 2 / §3.1 Image-Text Contrastive / §3.1 Image-Text Matching/LM
- **이 셀의 책임:** ITC·ITM·LM 세 동작 모드 / unimodal alignment / negative match·causal token prediction
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 세 loss 유한값 / 대칭 CE / binary/next-token loss. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO P-1: 논문의 modality 준비, 핵심 module, objective와 평가를 구현하세요.
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    hidden_dim: int = 16
    learning_rate: float = 0.02
    steps: int = 14


def prepare_views(image_batch: Tensor, text_batch: Tensor) -> tuple[Tensor, Tensor]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


class BlipPortfolioModel(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, config: PortfolioConfig):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, image_batch: Tensor, text_batch: Tensor) -> dict[str, Tensor]:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def compute_loss(self, outputs: dict[str, Tensor], text_batch: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def training_step(self, image_batch: Tensor, text_batch: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### Task P2 · objective, stop-gradient, update

positive/negative, online/target 또는 masked/visible 경계를 확인하면서
`fit_portfolio_model`의 gradient와 EMA update 순서를 추적합니다.

### 구현 단계 8 · 목적함수·학습 update

- **원문 위치:** §3.1, Figure 2
- **이 셀의 책임:** ITC·ITM·LM 세 동작 모드
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 세 loss 유한값. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(
    model: BlipPortfolioModel, image_batch: Tensor, text_batch: Tensor
) -> list[float]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### Task P3 · 표현 품질 평가

학습 loss와 retrieval·reconstruction·conditioning 지표를 분리해 계산합니다.
작은 로컬 pair의 수치는 원 논문의 downstream benchmark와 직접 비교하지 않습니다.

### 구현 단계 9 · 평가·완료 증거

- **원문 위치:** §3.1 Image-Text Contrastive / §3.1 Image-Text Matching/LM / §3.2, Figure 3
- **이 셀의 책임:** unimodal alignment / negative match·causal token prediction / captioner 생성+filter 제거 CapFilt
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 대칭 CE / binary/next-token loss / 정제 전후 noise. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: BlipPortfolioModel,
    image_batch: Tensor,
    text_batch: Tensor,
    keep_mask: Tensor,
) -> dict[str, float]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

## 실험 기록

1. 핵심 수식의 기호와 코드 변수 이름을 대응시키세요.
2. 어떤 assertion이 가장 중요한 구현 오류를 잡는지 설명하세요.
3. 이 미니 데이터로 검증할 수 없는 원 논문의 주장을 하나 적으세요.
4. seed를 유지하고 변수 하나만 바꾼 뒤 metric 전후를 기록하세요.